# BiLSTM + PPO Trader — Kaggle Training (V2)

Kaggle notebook: [https://www.kaggle.com/code/acaurangel/bilstm-ppo-self-attention-ai-spot-trading](https://www.kaggle.com/code/acaurangel/bilstm-ppo-self-attention-ai-spot-trading)

Full training pipeline for the BiLSTM (forecaster) and PPO (agent) models,
with improved Reward V2, vectorization for 2 GPUs, and TensorFlow.js export.

**Changes from V1:**
- Multi-exchange fallback (no Binance dependency — blocked on Kaggle)
- Multi-GPU via MirroredStrategy on 2× Tesla T4
- Vectorized episode runner (~12× faster than sequential)
- Reward V2: no idle penalty, profit bonus, entropy regularization
- Keras 3 compatible (`add_weight(name=, shape=)`)
- Automatic export to TensorFlow.js


In [ ]:
!pip install -q ccxt tensorflowjs

In [ ]:
import math, time, os, json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import ccxt

# ── Hyperparameters ──────────────────────────────────────────────────────────
WINDOW_SIZE          = 64
HORIZON              = 4
HIDDEN_UNITS         = 64
DROPOUT              = 0.2
NUM_FEATURES         = 10
L2                   = 0.0
LR                   = 5e-4
MIN_LR               = 1e-5
EARLY_STOP_PATIENCE  = 15
BATCH_SIZE           = 256
FORECAST_EPOCHS      = 100

# PPO
STATE_SIZE   = 13
ACTION_SIZE  = 3       # 0=HOLD 1=BUY 2=SELL
GAMMA        = 0.99
CLIP_RATIO   = 0.2
POLICY_LR    = 3e-4
VALUE_LR     = 1e-3
PPO_MIN_LR   = 1e-5
LR_DECAY     = 0.99
MAX_GRAD_NORM = 0.5
PPO_EPOCHS   = 10

# Constants
INDICATOR_WARMUP      = 200
RETURN_CLIP           = 0.1
VOLATILITY_WINDOW     = 20
FEE_RATE              = 0.000
BACKTEST_HOLDOUT      = 1000
VALIDATION_FRACTION   = 0.1
EMBARGO_FRACTION      = 0.01
HISTORICAL_CANDLES    = 50_000

# Risk
STOP_LOSS_PCT    = 0.015
TAKE_PROFIT_PCT  = 0.025
SLIPPAGE_PCT     = 0.0005

FORECASTER_SYMBOLS = ["BTC/USDT", "ETH/USDT", "DOGE/USDT", "XRP/USDT"]
AGENT_SYMBOL       = "BTC/USDT"
TIMEFRAME          = "1h"

# ── Multi-GPU ─────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs detected: {len(gpus)} -> {[g.name for g in gpus]}")

if len(gpus) > 1:
    strategy = tf.distribute.MirroredStrategy()
else:
    strategy = tf.distribute.get_strategy()

NUM_REPLICAS = strategy.num_replicas_in_sync
GLOBAL_BATCH = BATCH_SIZE * NUM_REPLICAS
print(f"Replicas: {NUM_REPLICAS}, global batch: {GLOBAL_BATCH}")


In [ ]:
# ── Technical indicators (pure NumPy) ────────────────────────────────────────

def ema(values: np.ndarray, period: int) -> np.ndarray:
    alpha = 2.0 / (period + 1)
    result = np.full(len(values), np.nan)
    if len(values) < period:
        return result
    result[period - 1] = np.mean(values[:period])
    for i in range(period, len(values)):
        result[i] = values[i] * alpha + result[i - 1] * (1 - alpha)
    return result

def rsi(values: np.ndarray, period: int = 14) -> np.ndarray:
    deltas = np.diff(values)
    result = np.full(len(values), np.nan)
    if len(deltas) < period:
        return result
    gains = np.where(deltas > 0, deltas, 0.0)
    losses = np.where(deltas < 0, -deltas, 0.0)
    avg_g, avg_l = np.mean(gains[:period]), np.mean(losses[:period])
    rs = avg_g / (avg_l + 1e-10)
    result[period] = 100 - 100 / (1 + rs)
    for i in range(period, len(deltas)):
        avg_g = (avg_g * (period - 1) + gains[i]) / period
        avg_l = (avg_l * (period - 1) + losses[i]) / period
        rs = avg_g / (avg_l + 1e-10)
        result[i + 1] = 100 - 100 / (1 + rs)
    return result

def macd(values: np.ndarray, fast=12, slow=26, signal=9):
    fast_ema = ema(values, fast)
    slow_ema = ema(values, slow)
    macd_line = fast_ema - slow_ema
    signal_line = ema(np.where(np.isnan(macd_line), 0, macd_line), signal)
    return macd_line, signal_line

def bollinger(values: np.ndarray, period: int = 20, std_dev: float = 2.0):
    upper = np.full(len(values), np.nan)
    lower = np.full(len(values), np.nan)
    for i in range(period - 1, len(values)):
        window = values[i - period + 1 : i + 1]
        m = np.mean(window)
        s = np.std(window, ddof=0)
        upper[i] = m + std_dev * s
        lower[i] = m - std_dev * s
    return upper, lower

print("Indicators OK.")


In [ ]:
# ── Candle fetch — multi-exchange with fallback ───────────────────────────────

TIMEFRAME_MS = {
    "1m": 60_000, "3m": 180_000, "5m": 300_000, "15m": 900_000,
    "30m": 1_800_000, "1h": 3_600_000, "2h": 7_200_000, "4h": 14_400_000,
    "6h": 21_600_000, "8h": 28_800_000, "12h": 43_200_000, "1d": 86_400_000,
}
MAX_PER_REQUEST = 1000

EXCHANGE_CONFIGS = [
    (lambda: ccxt.bybit({"enableRateLimit": True}),  {"category": "spot"}),
    (lambda: ccxt.okx({"enableRateLimit": True}),    {}),
    (lambda: ccxt.kucoin({"enableRateLimit": True}), {}),
    (lambda: ccxt.gate({"enableRateLimit": True}),   {}),
    (lambda: ccxt.mexc({"enableRateLimit": True}),   {}),
]

_active_exchange = None
_active_params   = {}

def _get_exchange():
    global _active_exchange, _active_params
    for factory, extra_params in EXCHANGE_CONFIGS:
        ex = factory()
        try:
            ex.load_markets()
            if "BTC/USDT" not in ex.markets:
                print(f"  x {ex.id} -- BTC/USDT not found")
                continue
            test = ex.fetch_ohlcv("BTC/USDT", "1h", limit=3, params=extra_params)
            if not test or len(test) == 0:
                print(f"  x {ex.id} -- fetch_ohlcv returned empty")
                continue
            print(f"  v Active exchange: {ex.id} (smoke-test OK: {len(test)} candles)")
            _active_exchange = ex
            _active_params   = extra_params
            return ex, extra_params
        except Exception as e:
            print(f"  x {ex.id} unavailable: {type(e).__name__}: {str(e)[:80]}")
    raise RuntimeError("No exchange accessible. Check Kaggle network connectivity.")

print("Searching for available exchange...")
_active_exchange, _active_params = _get_exchange()

# Diagnose available pairs
for sym in FORECASTER_SYMBOLS:
    available = sym in _active_exchange.markets
    print(f"  {sym}: {'OK' if available else 'NOT FOUND'}")

def fetch_candles(symbol: str, limit: int, timeframe: str = "1h") -> np.ndarray:
    """Returns array of shape (N, 6): [ts, open, high, low, close, volume]."""
    global _active_exchange, _active_params
    ex     = _active_exchange
    params = _active_params

    if symbol not in ex.markets:
        print(f"  ! {symbol} not found in {ex.id}")
        return np.empty((0, 6), dtype=np.float64)

    tf_ms      = TIMEFRAME_MS[timeframe]
    end_time   = int(time.time() * 1000)
    start_time = end_time - limit * tf_ms
    all_rows   = []
    since      = start_time

    while since < end_time:
        try:
            raw = ex.fetch_ohlcv(symbol, timeframe, since=since,
                                  limit=MAX_PER_REQUEST, params=params)
        except Exception as e:
            print(f"  ! Error on {ex.id} ({type(e).__name__}), reconnecting...")
            ex, params = _get_exchange()
            continue

        if not raw:
            break

        filtered = [row for row in raw if row[0] <= end_time]
        all_rows.extend(filtered)

        # Advance cursor by timestamp, NOT by batch size
        next_since = raw[-1][0] + tf_ms
        if next_since <= since:
            break
        since = next_since

        if len(all_rows) >= limit:
            break

        time.sleep(ex.rateLimit / 1000)

    if len(all_rows) == 0:
        print(f"  ! WARNING: 0 candles returned for {symbol}!")
        return np.empty((0, 6), dtype=np.float64)

    arr = np.array(all_rows, dtype=np.float64)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    _, idx = np.unique(arr[:, 0], return_index=True)
    arr = arr[idx]

    print(f"  -> {symbol}: {len(arr)} candles downloaded (requested: {limit})")
    return arr[-limit:]

print("Fetch function ready.")


In [ ]:
# ── Feature builder (10 features per timestep) ───────────────────────────────

def build_features_for_series(candles: np.ndarray, window_size: int = 64) -> tuple:
    closes  = candles[:, 4]
    opens   = candles[:, 1]
    highs   = candles[:, 2]
    lows    = candles[:, 3]
    volumes = candles[:, 5]

    ema9_all   = ema(closes, 9)
    ema21_all  = ema(closes, 21)
    rsi14_all  = rsi(closes, 14)
    macd_line, _ = macd(closes, 12, 26, 9)
    bb_upper, bb_lower = bollinger(closes, 20, 2.0)

    X_list, y_list = [], []
    n = len(candles)

    for i in range(window_size, n - HORIZON):
        w_start = i - window_size
        w_end   = i

        ref_close  = closes[w_start]
        max_volume = np.max(volumes[w_start:w_end]) if np.max(volumes[w_start:w_end]) > 0 else 1.0

        rows = []
        for j in range(w_start, w_end):
            c = closes[j]
            prev_ret = 0.0 if j == w_start else (c - closes[j-1]) / closes[j-1]
            r9   = ema9_all[j]   if not np.isnan(ema9_all[j])   else c
            r21  = ema21_all[j]  if not np.isnan(ema21_all[j])  else c
            rsi_v = rsi14_all[j] if not np.isnan(rsi14_all[j])  else 50.0
            macd_v = macd_line[j] if not np.isnan(macd_line[j]) else 0.0
            bbu  = bb_upper[j]   if not np.isnan(bb_upper[j])   else c
            bbl  = bb_lower[j]   if not np.isnan(bb_lower[j])   else c
            rows.append([
                c / ref_close - 1,
                volumes[j] / max_volume,
                rsi_v / 100.0,
                macd_v / c,
                prev_ret,
                (highs[j] - lows[j]) / c,
                (r9 - c) / c,
                (r21 - c) / c,
                (bbu - c) / c,
                (bbl - c) / c,
            ])

        X_list.append(rows)

        anchor_close = closes[i]
        future = []
        for h in range(1, HORIZON + 1):
            raw_ret = (closes[i + h] - anchor_close) / anchor_close
            future.append(np.clip(raw_ret, -RETURN_CLIP, RETURN_CLIP))
        y_list.append(future)

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


def compute_volatility(candles: np.ndarray) -> np.ndarray:
    closes = candles[:, 4]
    n = len(closes)
    vol = np.zeros(n)
    for i in range(1, n):
        start = max(1, i - VOLATILITY_WINDOW)
        rets = (closes[start:i+1] - closes[start-1:i]) / closes[start-1:i]
        vol[i] = np.std(rets)
    return vol

print("Feature builder OK.")


In [ ]:
# ── Purged Embargoed Split ───────────────────────────────────────────────────

def purged_embargoed_split(X, y, horizon, embargo_frac, val_frac=0.1):
    total = len(X)
    val_size   = int(total * val_frac)
    val_start  = total - val_size
    embargo_sz = int(total * embargo_frac)
    train_end  = max(0, val_start - horizon - embargo_sz)
    purged = val_start - train_end
    return (
        X[:train_end], y[:train_end],
        X[val_start:],  y[val_start:],
        purged
    )

print("Split OK.")


In [ ]:
# ── Download and preprocessing ───────────────────────────────────────────────

total_needed = HISTORICAL_CANDLES + BACKTEST_HOLDOUT

all_X_train, all_y_train = [], []
all_X_val,   all_y_val   = [], []
agent_candles = None

for sym in FORECASTER_SYMBOLS:
    print(f"\nDownloading {total_needed} candles for {sym}...")
    raw = fetch_candles(sym, total_needed, TIMEFRAME)
    train_raw = raw[:-BACKTEST_HOLDOUT] if len(raw) > BACKTEST_HOLDOUT else raw

    print(f"  {len(train_raw)} candles for training")

    min_needed = WINDOW_SIZE + HORIZON + 1
    if len(train_raw) < min_needed:
        print(f"  ! Insufficient data (minimum: {min_needed}) -- skipping.")
        continue

    if sym == AGENT_SYMBOL:
        agent_candles = train_raw.copy()

    X, y = build_features_for_series(train_raw, WINDOW_SIZE)
    Xtr, ytr, Xv, yv, purged = purged_embargoed_split(
        X, y, HORIZON, EMBARGO_FRACTION, VALIDATION_FRACTION
    )
    all_X_train.append(Xtr);  all_y_train.append(ytr)
    all_X_val.append(Xv);     all_y_val.append(yv)
    print(f"  train={len(Xtr)}, val={len(Xv)}, purged={purged}")

if not all_X_train:
    raise RuntimeError("No symbol returned data.")

if agent_candles is None:
    raise RuntimeError(f"No data for agent ({AGENT_SYMBOL}).")

X_train = np.concatenate(all_X_train, axis=0)
y_train = np.concatenate(all_y_train, axis=0)
X_val   = np.concatenate(all_X_val,   axis=0)
y_val   = np.concatenate(all_y_val,   axis=0)

print(f"\nTotal dataset -- train: {X_train.shape}, val: {X_val.shape}")


In [ ]:
# ── Pre-compute agent features and volatility ────────────────────────────────

print("Pre-computing agent features...")
agent_X, _ = build_features_for_series(agent_candles, WINDOW_SIZE)
agent_vol  = compute_volatility(agent_candles)
agent_vol_samples = agent_vol[WINDOW_SIZE : len(agent_candles) - HORIZON]

print(f"Agent features: {agent_X.shape}, volatility: {agent_vol_samples.shape}")


In [ ]:
# ── SelfAttentionLayer (Keras 3 compatible) ──────────────────────────────────

class SelfAttentionLayer(layers.Layer):
    """Scaled dot-product self-attention, single-head."""
    def __init__(self, d_model: int, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model

    def build(self, input_shape):
        feat_dim = input_shape[-1]
        init = keras.initializers.GlorotUniform()
        self.Wq = self.add_weight(name="wq", shape=(feat_dim, self.d_model), initializer=init)
        self.Wk = self.add_weight(name="wk", shape=(feat_dim, self.d_model), initializer=init)
        self.Wv = self.add_weight(name="wv", shape=(feat_dim, self.d_model), initializer=init)
        super().build(input_shape)

    def call(self, x):
        Q = tf.matmul(x, self.Wq)
        K = tf.matmul(x, self.Wk)
        V = tf.matmul(x, self.Wv)
        scale   = math.sqrt(self.d_model)
        scores  = tf.matmul(Q, K, transpose_b=True) / scale
        weights = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(weights, V)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"d_model": self.d_model})
        return cfg

print("SelfAttentionLayer OK.")


In [ ]:
# ── BiLSTM (inside strategy.scope for multi-GPU) ────────────────────────────

def build_bilstm(seq_len=64, num_features=10, hidden=64,
                 dropout=0.2, horizon=4, l2_val=1e-4):
    reg = regularizers.L2(l2_val)
    inp = keras.Input(shape=(seq_len, num_features))

    x = layers.Bidirectional(
        layers.LSTM(hidden, return_sequences=True,
                    kernel_regularizer=reg, recurrent_regularizer=reg)
    )(inp)
    x = layers.Dropout(dropout)(x)

    x = layers.Bidirectional(
        layers.LSTM(hidden // 2, return_sequences=True,
                    kernel_regularizer=reg, recurrent_regularizer=reg)
    )(x)
    x = layers.Dropout(dropout)(x)

    x = SelfAttentionLayer(d_model=hidden)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(hidden, kernel_regularizer=reg,
                     bias_initializer=keras.initializers.Constant(0.01))(x)
    x = layers.LeakyReLU(negative_slope=0.01)(x)
    out = layers.Dense(horizon)(x)

    return keras.Model(inputs=inp, outputs=out, name="BiLSTM_Forecaster")

with strategy.scope():
    forecaster = build_bilstm(
        WINDOW_SIZE, NUM_FEATURES, HIDDEN_UNITS, DROPOUT, HORIZON, L2
    )

forecaster.summary()


In [ ]:
# ── BiLSTM training (cosine annealing + early stopping) ─────────────────────

class CosineAnnealingCallback(keras.callbacks.Callback):
    def __init__(self, lr_max, lr_min, total_epochs):
        super().__init__()
        self.lr_max = lr_max
        self.lr_min = lr_min
        self.total  = total_epochs

    def on_epoch_begin(self, epoch, logs=None):
        cosine = 0.5 * (1 + math.cos(math.pi * epoch / self.total))
        lr = self.lr_min + (self.lr_max - self.lr_min) * cosine
        self.model.optimizer.learning_rate.assign(lr)


with strategy.scope():
    forecaster.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss="mae",
        metrics=["mse"]
    )

callbacks = [
    CosineAnnealingCallback(LR, MIN_LR, FORECAST_EPOCHS),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=EARLY_STOP_PATIENCE,
        min_delta=1e-6, restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        "/kaggle/working/bilstm_best.keras",
        monitor="val_loss", save_best_only=True, verbose=1
    ),
]

print(f"Training BiLSTM -- {FORECAST_EPOCHS} max epochs (batch={GLOBAL_BATCH})...")
history = forecaster.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=FORECAST_EPOCHS,
    batch_size=GLOBAL_BATCH,
    shuffle=True,
    callbacks=callbacks,
    verbose=1,
)
print("BiLSTM trained.")


In [ ]:
# ── Pre-compute agent forecasts in batch ─────────────────────────────────────

CHUNK = 512
agent_forecasts = []
n_samples = len(agent_X)
print(f"Generating {n_samples} forecasts in batch...")
for start in range(0, n_samples, CHUNK):
    batch = agent_X[start : start + CHUNK]
    preds = forecaster.predict(batch, verbose=0)
    agent_forecasts.append(preds)
agent_forecasts = np.concatenate(agent_forecasts, axis=0)
print(f"Pre-computed forecasts: {agent_forecasts.shape}")


## PPO — Reward V2 + Multi-GPU + Vectorization

Changes from the original pipeline:
- **Reward V2**: no idle penalty; profit bonus; forecaster-based opportunity cost; progressive hold-loser penalty
- **Entropy bonus** (0.01) on actor loss to prevent policy collapse
- **Vectorization**: 16 parallel episodes per update (~12x speedup over sequential)
- **Best checkpoint**: automatically saves the best model during training


In [ ]:
# ── Reward V2 + helper functions ─────────────────────────────────────────────

# Reward V2 hyperparameters
PROFIT_BONUS       = 3.0    # extra multiplier for profitable sells
OPPORTUNITY_COST   = 0.3    # weight of opportunity cost when flat
HOLD_WINNER_BONUS  = 1.5    # multiplier when holding a winning position
HOLD_LOSER_DECAY   = 0.05   # growing penalty for holding a losing position
LOSS_CUT_BONUS     = 0.001  # bonus for cutting a loss before stop-loss

def assemble_state(candles, i, forecast, position, entry_price, bars_in_pos, volatility):
    close = candles[i, 4]
    open_ = candles[i, 1]
    high  = candles[i, 2]
    low   = candles[i, 3]
    pnl   = (close - entry_price) / entry_price if position == 1 and entry_price > 0 else 0.0
    return [
        1 - position, pnl, min(bars_in_pos / 100.0, 1.0),
        (open_ - close) / close, (high - close) / close,
        (low - close) / close, (high - low) / close,
        position, volatility, *forecast[:4],
    ]

def apply_exit_guards(action, position, entry_price, current_close, sl, tp):
    if position != 1:
        return action
    pnl = (current_close - entry_price) / entry_price if entry_price > 0 else 0.0
    if pnl <= -sl or pnl >= tp:
        return 2
    return action

def compute_reward_v2(candles, i, action, position, entry_price, bars_in_pos, forecast):
    """Reward V2 -- no idle penalty; shaped by forecaster output."""
    close      = candles[i, 4]
    next_close = candles[i + 1, 4]
    price_ret  = (next_close - close) / close
    pred_ret   = forecast[0]

    # FLAT (no position)
    if position == 0:
        if action == 1:  # BUY
            base = price_ret - FEE_RATE - SLIPPAGE_PCT
            if pred_ret > 0:
                base += abs(pred_ret) * OPPORTUNITY_COST
            return base
        else:  # HOLD flat
            if pred_ret > 0.002:
                return -abs(pred_ret) * OPPORTUNITY_COST * 0.3
            return 0.0

    # IN POSITION
    pnl = (close - entry_price) / entry_price if entry_price > 0 else 0.0

    if action == 2:  # SELL
        realized = pnl
        base = realized - 2 * FEE_RATE - 2 * SLIPPAGE_PCT
        if realized > 0:
            base += realized * PROFIT_BONUS
        elif realized < -0.005 and realized > -STOP_LOSS_PCT:
            base += LOSS_CUT_BONUS
        return base

    # HOLD in position
    if pnl > 0:
        return price_ret * HOLD_WINNER_BONUS
    else:
        time_factor = min(bars_in_pos / 20.0, 1.0)
        hold_penalty = -abs(pnl) * HOLD_LOSER_DECAY * time_factor
        return price_ret + hold_penalty

print("Reward V2 + helper functions OK.")
print(f"  Profit bonus: {PROFIT_BONUS}x | Opportunity cost: {OPPORTUNITY_COST}")
print(f"  Hold winner: {HOLD_WINNER_BONUS}x | Hold loser decay: {HOLD_LOSER_DECAY}")


In [ ]:
# ── Actor / Critic (inside strategy.scope, optimizers pre-built) ─────────────

def build_actor(state_size=13, action_size=3):
    inp = keras.Input(shape=(state_size,))
    x   = layers.Dense(128, activation="tanh")(inp)
    x   = layers.Dense(64,  activation="tanh")(x)
    out = layers.Dense(action_size, activation="softmax")(x)
    return keras.Model(inp, out, name="Actor")

def build_critic(state_size=13):
    inp = keras.Input(shape=(state_size,)
)
    x   = layers.Dense(128, activation="tanh")(inp)
    x   = layers.Dense(64,  activation="tanh")(x)
    out = layers.Dense(1)(x)
    return keras.Model(inp, out, name="Critic")

with strategy.scope():
    actor  = build_actor(STATE_SIZE, ACTION_SIZE)
    critic = build_critic(STATE_SIZE)
    policy_opt = keras.optimizers.Adam(POLICY_LR)
    value_opt  = keras.optimizers.Adam(VALUE_LR)
    policy_opt.build(actor.trainable_variables)
    value_opt.build(critic.trainable_variables)

update_count = 0

def get_current_lr(base_lr, decay, count):
    return max(base_lr * (decay ** count), PPO_MIN_LR)

def sample_action(probs):
    return np.random.choice(len(probs), p=probs)

def decide(state_vec):
    s = tf.convert_to_tensor([state_vec], dtype=tf.float32)
    probs = actor(s, training=False).numpy()[0]
    value = critic(s, training=False).numpy()[0, 0]
    action = sample_action(probs)
    log_prob = math.log(probs[action] + 1e-8)
    return action, log_prob, value

print("Actor/Critic OK (built inside strategy scope).")
actor.summary()


In [ ]:
# ── PPO update functions with entropy bonus ───────────────────────────────────

ENTROPY_COEFF = 0.01

def compute_advantages(rewards, values, terminals, gamma=0.99, lam=0.95):
    n = len(rewards)
    advantages = np.zeros(n)
    last_adv = 0.0
    for t in reversed(range(n)):
        mask     = 0.0 if terminals[t] else 1.0
        next_val = values[t+1] if t + 1 < n else 0.0
        delta    = rewards[t] + gamma * next_val * mask - values[t]
        last_adv = delta + gamma * lam * mask * last_adv
        advantages[t] = last_adv
    mean, std = advantages.mean(), advantages.std() + 1e-8
    return (advantages - mean) / std

def compute_returns(rewards, terminals, gamma=0.99):
    n = len(rewards)
    returns = np.zeros(n)
    last = 0.0
    for t in reversed(range(n)):
        mask   = 0.0 if terminals[t] else 1.0
        last   = rewards[t] + gamma * mask * last
        returns[t] = last
    return returns

def clip_grads(grads, max_norm):
    global_norm = tf.linalg.global_norm(grads)
    scale = tf.minimum(1.0, max_norm / (global_norm + 1e-6))
    return [g * scale for g in grads]

@tf.function
def update_actor(states, actions, advantages, old_log_probs):
    with tf.GradientTape() as tape:
        probs     = actor(states, training=True)
        log_probs = tf.math.log(probs + 1e-8)
        indices   = tf.stack([tf.range(tf.shape(actions)[0]), actions], axis=1)
        new_lp    = tf.gather_nd(log_probs, indices)
        ratio     = tf.exp(new_lp - old_log_probs)
        surr1     = ratio * advantages
        surr2     = tf.clip_by_value(ratio, 1 - CLIP_RATIO, 1 + CLIP_RATIO) * advantages
        policy_loss = -tf.reduce_mean(tf.minimum(surr1, surr2))
        entropy = -tf.reduce_mean(tf.reduce_sum(probs * log_probs, axis=1))
        loss = policy_loss - ENTROPY_COEFF * entropy
    grads = tape.gradient(loss, actor.trainable_variables)
    grads = clip_grads(grads, MAX_GRAD_NORM)
    policy_opt.apply_gradients(zip(grads, actor.trainable_variables))
    return loss, entropy

@tf.function
def update_critic(states, returns):
    with tf.GradientTape() as tape:
        values = tf.squeeze(critic(states, training=True), axis=1)
        loss   = tf.reduce_mean(tf.square(returns - values))
    grads = tape.gradient(loss, critic.trainable_variables)
    grads = clip_grads(grads, MAX_GRAD_NORM)
    value_opt.apply_gradients(zip(grads, critic.trainable_variables))
    return loss

def ppo_update(states_np, actions_np, log_probs_np, values_np, rewards_np, terminals_np):
    global update_count
    advantages  = compute_advantages(rewards_np, values_np, terminals_np, GAMMA)
    returns_arr = compute_returns(rewards_np, terminals_np, GAMMA)
    
    states     = tf.constant(states_np, dtype=tf.float32)
    actions    = tf.constant(actions_np, dtype=tf.int32)
    old_lp     = tf.constant(log_probs_np, dtype=tf.float32)
    adv_tensor = tf.constant(advantages,  dtype=tf.float32)
    ret_tensor = tf.constant(returns_arr, dtype=tf.float32)
    
    for _ in range(PPO_EPOCHS):
        update_actor(states, actions, adv_tensor, old_lp)
        update_critic(states, ret_tensor)
        
    update_count += 1
    new_p_lr = get_current_lr(POLICY_LR, LR_DECAY, update_count)
    new_v_lr = get_current_lr(VALUE_LR,  LR_DECAY, update_count)
    policy_opt.learning_rate.assign(new_p_lr)
    value_opt.learning_rate.assign(new_v_lr)

print(f"PPO update functions OK. (entropy_coeff={ENTROPY_COEFF})")


In [ ]:
# ── Vectorized episode runner V2 (PURE TF GPU) ──────────────────────────────

NUM_PARALLEL = 16

@tf.function
def tf_run_episodes_vectorized(closes, opens, highs, lows, forecasts, vol_samples, n_envs, n_steps):
    states_ta    = tf.TensorArray(tf.float32, size=n_steps)
    actions_ta   = tf.TensorArray(tf.int32,   size=n_steps)
    log_probs_ta = tf.TensorArray(tf.float32, size=n_steps)
    values_ta    = tf.TensorArray(tf.float32, size=n_steps)
    rewards_ta   = tf.TensorArray(tf.float32, size=n_steps)

    positions    = tf.zeros([n_envs], dtype=tf.int32)
    entry_prices = tf.zeros([n_envs], dtype=tf.float32)
    bars         = tf.zeros([n_envs], dtype=tf.int32)

    for idx in tf.range(n_steps):
        i = idx + WINDOW_SIZE
        
        close = closes[i]
        open_ = opens[i]
        high  = highs[i]
        low   = lows[i]
        vol   = vol_samples[idx]
        forecast = forecasts[idx]

        close_batch = tf.fill([n_envs], close)
        open_batch  = tf.fill([n_envs], open_)
        high_batch  = tf.fill([n_envs], high)
        low_batch   = tf.fill([n_envs], low)
        vol_batch   = tf.fill([n_envs], vol)
        forecast_batch = tf.tile(tf.expand_dims(forecast[:4], 0), [n_envs, 1])

        safe_entry = tf.where(entry_prices > 0.0, entry_prices, 1.0)
        pnls = tf.where((positions == 1) & (entry_prices > 0.0), (close_batch - safe_entry) / safe_entry, 0.0)

        st_1 = tf.cast(1 - positions, tf.float32)
        st_2 = pnls
        st_3 = tf.minimum(tf.cast(bars, tf.float32) / 100.0, 1.0)
        st_4 = (open_batch - close_batch) / close_batch
        st_5 = (high_batch - close_batch) / close_batch
        st_6 = (low_batch - close_batch) / close_batch
        st_7 = (high_batch - low_batch) / close_batch
        st_8 = tf.cast(positions, tf.float32)
        st_9 = vol_batch

        states = tf.concat([
            tf.expand_dims(st_1, 1), tf.expand_dims(st_2, 1), tf.expand_dims(st_3, 1),
            tf.expand_dims(st_4, 1), tf.expand_dims(st_5, 1), tf.expand_dims(st_6, 1),
            tf.expand_dims(st_7, 1), tf.expand_dims(st_8, 1), tf.expand_dims(st_9, 1),
            forecast_batch
        ], axis=1)

        probs  = actor(states, training=False)
        values = tf.squeeze(critic(states, training=False), axis=1)

        # Gumbel-max trick for stable random categorical sampling on GPU
        u = tf.random.uniform(tf.shape(probs), minval=0.00001, maxval=0.99999)
        actions = tf.argmax(tf.math.log(probs + 1e-8) - tf.math.log(-tf.math.log(u)), axis=-1, output_type=tf.int32)

        indices = tf.stack([tf.range(n_envs), actions], axis=1)
        log_probs = tf.math.log(tf.gather_nd(probs, indices) + 1e-8)

        force_sell = (positions == 1) & ((pnls <= -STOP_LOSS_PCT) | (pnls >= TAKE_PROFIT_PCT))
        effective_actions = tf.where(force_sell, 2, actions)

        next_close = closes[i + 1]
        price_ret  = (next_close - close) / close
        pred_ret   = forecast[0]

        rewards = tf.zeros([n_envs], dtype=tf.float32)
        
        is_flat   = (positions == 0)
        flat_buy  = is_flat & (effective_actions == 1)
        flat_hold = is_flat & (effective_actions != 1)
        
        is_in     = (positions == 1)
        in_sell   = is_in & (effective_actions == 2)
        in_hold   = is_in & (effective_actions != 2)

        base_buy = price_ret - FEE_RATE - SLIPPAGE_PCT
        base_buy = tf.where(pred_ret > 0.0, base_buy + tf.abs(pred_ret) * OPPORTUNITY_COST, base_buy)
        base_hold = tf.where(pred_ret > 0.002, -tf.abs(pred_ret) * OPPORTUNITY_COST * 0.3, 0.0)

        realized = pnls
        base_sell = realized - 2.0 * FEE_RATE - 2.0 * SLIPPAGE_PCT
        base_sell = tf.where(realized > 0.0, base_sell + realized * PROFIT_BONUS, base_sell)
        base_sell = tf.where((realized < -0.005) & (realized > -STOP_LOSS_PCT), base_sell + LOSS_CUT_BONUS, base_sell)

        time_factor = tf.minimum(tf.cast(bars, tf.float32) / 20.0, 1.0)
        hold_penalty = -tf.abs(pnls) * HOLD_LOSER_DECAY * time_factor
        base_in_hold = tf.where(pnls > 0.0, price_ret * HOLD_WINNER_BONUS, price_ret + hold_penalty)

        rewards = tf.where(flat_buy, base_buy, rewards)
        rewards = tf.where(flat_hold, base_hold, rewards)
        rewards = tf.where(in_sell, base_sell, rewards)
        rewards = tf.where(in_hold, base_in_hold, rewards)

        positions = tf.where(flat_buy, 1, positions)
        positions = tf.where(in_sell,  0, positions)

        entry_prices = tf.where(flat_buy, close_batch, entry_prices)
        entry_prices = tf.where(in_sell,  0.0,         entry_prices)

        bars = tf.where(flat_buy, 0, bars)
        bars = tf.where(in_sell,  0, bars)
        bars = tf.where(in_hold,  bars + 1, bars)

        states_ta    = states_ta.write(idx, states)
        actions_ta   = actions_ta.write(idx, actions)
        log_probs_ta = log_probs_ta.write(idx, log_probs)
        values_ta    = values_ta.write(idx, values)
        rewards_ta   = rewards_ta.write(idx, rewards)

    return states_ta.stack(), actions_ta.stack(), log_probs_ta.stack(), values_ta.stack(), rewards_ta.stack()

def run_episodes_vectorized(candles, forecasts, vol_samples, n_envs=NUM_PARALLEL):
    n_steps = len(forecasts)
    closes = tf.constant(candles[:, 4], dtype=tf.float32)
    opens  = tf.constant(candles[:, 1], dtype=tf.float32)
    highs  = tf.constant(candles[:, 2], dtype=tf.float32)
    lows   = tf.constant(candles[:, 3], dtype=tf.float32)
    f_cast = tf.constant(forecasts, dtype=tf.float32)
    v_samp = tf.constant(vol_samples, dtype=tf.float32)

    st, ac, lp, va, rw = tf_run_episodes_vectorized(
        closes, opens, highs, lows, f_cast, v_samp, 
        tf.constant(n_envs, dtype=tf.int32), 
        tf.constant(n_steps, dtype=tf.int32)
    )
    
    st_np = st.numpy()
    ac_np = ac.numpy()
    lp_np = lp.numpy()
    va_np = va.numpy()
    rw_np = rw.numpy()
    
    terminals = np.zeros(n_steps, dtype=bool)
    terminals[-1] = True
    all_experiences = []
    for e in range(n_envs):
        all_experiences.append((st_np[:, e, :], ac_np[:, e], lp_np[:, e], va_np[:, e], rw_np[:, e], terminals))
    return all_experiences

print("Vectorized episode runner V2 (PURE TF GPU + Numpy Slicing) OK.")


In [ ]:
# ── PPO V2 training loop with detailed metrics + best checkpoint ──────────────

import os
os.makedirs("/kaggle/working/models/ppo/policy", exist_ok=True)
os.makedirs("/kaggle/working/models/ppo/value",  exist_ok=True)

TOTAL_UPDATES = 20

print(f"Training PPO V2 -- {TOTAL_UPDATES} updates x {NUM_PARALLEL} parallel episodes...")
print(f"Reward: no idle penalty | profit_bonus={PROFIT_BONUS} | entropy={ENTROPY_COEFF}")
print()

best_avg_reward = -float('inf')

for update in range(TOTAL_UPDATES):
    t0 = time.time()
    batch_experiences = run_episodes_vectorized(
        agent_candles, agent_forecasts, agent_vol_samples, NUM_PARALLEL
    )
    total_reward = 0
    total_buys   = 0
    total_sells  = 0
    
    for ep_exp in batch_experiences:
        st, ac, lp, va, rw, term = ep_exp
        ppo_update(st, ac, lp, va, rw, term)
        total_reward += np.sum(rw)
        total_buys   += np.sum(ac == 1)
        total_sells  += np.sum(ac == 2)
        
    elapsed    = time.time() - t0
    avg_reward = total_reward / NUM_PARALLEL
    avg_buys   = total_buys  / NUM_PARALLEL
    avg_sells  = total_sells / NUM_PARALLEL
    marker = ""
    if avg_reward > best_avg_reward:
        best_avg_reward = avg_reward
        marker = " * BEST"
        actor.save("/kaggle/working/models/ppo/policy/best_model.keras")
        critic.save("/kaggle/working/models/ppo/value/best_model.keras")
    print(f"  Update {update+1:2d}/{TOTAL_UPDATES} | "
          f"avg_reward={avg_reward:+.4f} | "
          f"buys={avg_buys:.0f} sells={avg_sells:.0f} | "
          f"{elapsed:.0f}s{marker}")

print()
print(f"PPO V2 trained. Best avg_reward: {best_avg_reward:+.4f}")


In [ ]:
# ── Save final models (PPO uses best checkpoint) ─────────────────────────────

import os, shutil
os.makedirs("/kaggle/working/models/bilstm", exist_ok=True)
os.makedirs("/kaggle/working/models/ppo/policy", exist_ok=True)
os.makedirs("/kaggle/working/models/ppo/value",  exist_ok=True)

# BiLSTM: save current state (already has best weights via restore_best_weights)
forecaster.save("/kaggle/working/models/bilstm/model.keras")

# PPO: use the best checkpoint saved during training
shutil.copy("/kaggle/working/models/ppo/policy/best_model.keras",
            "/kaggle/working/models/ppo/policy/model.keras")
shutil.copy("/kaggle/working/models/ppo/value/best_model.keras",
            "/kaggle/working/models/ppo/value/model.keras")

print("Models saved in /kaggle/working/models/")
print("  bilstm/model.keras")
print("  ppo/policy/model.keras  (best checkpoint)")
print("  ppo/value/model.keras   (best checkpoint)")


## Export to TensorFlow.js

All three models are exported as **`tfjs_graph_model`** (frozen computation graph).
This avoids the Keras 3 layers-model format that `tfjs-layers` (v4.x) cannot deserialize
(error: `Corrupted configuration, expected array for nodeData`).

Graph models are inference-only, which is fine here — training happens on Kaggle, the
bot only runs `predict()` in Node.

The final `tfjs_models.zip` contains everything needed to run the bot:
```
tfjs_models/
  bilstm/model.json        <- graph model (SelfAttentionLayer baked in)
  ppo/
    policy/model.json      <- actor (graph model)
    value/model.json       <- critic (graph model)
```
**Setup:** download `tfjs_models.zip` from the Kaggle output panel and extract into `./models/`.


In [ ]:
# ── Export to TensorFlow.js (all models as graph_model) ──────────────────────

import shutil, subprocess

OUTPUT_BASE = "/kaggle/working/tfjs_models"
os.makedirs(OUTPUT_BASE, exist_ok=True)

# BiLSTM: rebuild on CPU and export as SavedModel (avoids CudnnRNN incompatibility)
cpu_keras_path = "/kaggle/working/models/bilstm/cpu_model.keras"
cpu_saved_path = "/kaggle/working/models/bilstm/cpu_saved_model"
cpu_forecaster = build_bilstm(WINDOW_SIZE, NUM_FEATURES, HIDDEN_UNITS, DROPOUT, HORIZON, L2)
cpu_forecaster.set_weights(forecaster.get_weights())
cpu_forecaster.save(cpu_keras_path)
print("BiLSTM saved as .keras (CPU copy)")

# Inline script runs conversion without GPU to avoid CudnnRNN in the SavedModel
convert_script = f"""
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import math, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class SelfAttentionLayer(layers.Layer):
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
    def build(self, input_shape):
        feat_dim = input_shape[-1]
        init = keras.initializers.GlorotUniform()
        self.Wq = self.add_weight(name='wq', shape=(feat_dim, self.d_model), initializer=init)
        self.Wk = self.add_weight(name='wk', shape=(feat_dim, self.d_model), initializer=init)
        self.Wv = self.add_weight(name='wv', shape=(feat_dim, self.d_model), initializer=init)
        super().build(input_shape)
    def call(self, x):
        Q, K, V = tf.matmul(x, self.Wq), tf.matmul(x, self.Wk), tf.matmul(x, self.Wv)
        return tf.matmul(tf.nn.softmax(tf.matmul(Q, K, transpose_b=True) / math.sqrt(self.d_model), -1), V)
    def get_config(self):
        cfg = super().get_config()
        cfg['d_model'] = self.d_model
        return cfg

model = keras.models.load_model('{cpu_keras_path}', custom_objects={{'SelfAttentionLayer': SelfAttentionLayer}})
model.export('{cpu_saved_path}')
print('SavedModel exported on CPU (no CudnnRNN)')
"""

with open('/tmp/convert_bilstm.py', 'w') as f:
    f.write(convert_script)

result = subprocess.run(
    ['python', '/tmp/convert_bilstm.py'],
    capture_output=True, text=True,
    env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''}
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])

# Convert BiLSTM SavedModel -> tfjs_graph_model
result2 = subprocess.run([
    'tensorflowjs_converter',
    '--input_format=tf_saved_model',
    '--output_format=tfjs_graph_model',
    '--signature_name=serving_default',
    cpu_saved_path,
    f'{OUTPUT_BASE}/bilstm'
], capture_output=True, text=True,
   env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''})
print(result2.stdout)
if result2.returncode != 0:
    print('STDERR:', result2.stderr[-500:])
else:
    print('BiLSTM converted to tfjs_graph_model')
    # --- PATCH: Remove control dependencies from TF.js model.json ---
    # TF.js GraphExecutor fails on ^ dependencies inside while loops for BiLSTM
    model_json_path = f'{OUTPUT_BASE}/bilstm/model.json'
    with open(model_json_path, 'r') as f:
        m_data = json.load(f)
    for node in m_data.get('modelTopology', {}).get('node', []):
        if 'input' in node:
            node['input'] = [i for i in node['input'] if not i.startswith('^')]
    with open(model_json_path, 'w') as f:
        json.dump(m_data, f)
    print('Control dependencies stripped from BiLSTM model.json')

# ── Actor / Critic: SavedModel -> tfjs_graph_model ───────────────────────────
# Avoids the Keras 3 layers-model format that tfjs-layers cannot deserialize.
# Net effect on the Node side: load with tf.loadGraphModel instead of
# tf.loadLayersModel. .predict() works identically (inference only — fine
# because PPO trains on Kaggle and the bot only runs inference).

def export_keras_to_tfjs_graph(model_obj, name):
    saved_path = f"/kaggle/working/models/ppo/{name}/saved_model"
    if os.path.exists(saved_path):
        shutil.rmtree(saved_path)
    model_obj.export(saved_path)

    out_path = f"{OUTPUT_BASE}/ppo/{name}"
    res = subprocess.run([
        'tensorflowjs_converter',
        '--input_format=tf_saved_model',
        '--output_format=tfjs_graph_model',
        '--signature_name=serving_default',
        saved_path,
        out_path
    ], capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print(f'STDERR ({name}):', res.stderr[-500:])
        return False
    print(f'  {name} -> tfjs_graph_model OK')
    return True

export_keras_to_tfjs_graph(actor,  'policy')
export_keras_to_tfjs_graph(critic, 'value')

# Sanity check: confirm the three model.json files declare graph-model format.
for sub in ['bilstm', 'ppo/policy', 'ppo/value']:
    with open(f'{OUTPUT_BASE}/{sub}/model.json', 'r') as f:
        fmt = json.load(f).get('format', '<missing>')
    print(f'  {sub}: format = {fmt}')
    assert fmt == 'graph-model', f'{sub} expected graph-model, got {fmt}'

# Package everything
shutil.make_archive('/kaggle/working/tfjs_models', 'zip', OUTPUT_BASE)
print('\nFinal file: /kaggle/working/tfjs_models.zip')
subprocess.run(['find', OUTPUT_BASE, '-type', 'f'], check=True)


## Loading in TypeScript

Kaggle notebook: [https://www.kaggle.com/code/acaurangel/bilstm-ppo-self-attention-ai-spot-trading](https://www.kaggle.com/code/acaurangel/bilstm-ppo-self-attention-ai-spot-trading)

**Setup:**
1. Download `tfjs_models.zip` from the Kaggle output panel after training.
2. Extract into `./models/` at the project root.

**Model formats:** all three are `tfjs_graph_model`, loaded with `tf.loadGraphModel`.

```typescript
const forecaster = await tf.loadGraphModel('file://./models/bilstm/model.json');
const actor      = await tf.loadGraphModel('file://./models/ppo/policy/model.json');
const critic     = await tf.loadGraphModel('file://./models/ppo/value/model.json');
```
